# Session 8 · Case study: first leaderboard submission (round L1)

*Author: course team · Licence: CC-BY-4.0*

Practice task of block 3: a logistic regression on the simple description features of Session 2 plus the language and the issuing country, validated by time, written as `submission.csv` with the columns `id,heading` (heading as a four-digit string).

The task of the leaderboard: predict the four-digit HS heading of each test decision (2024–2026) from what a trader's request contains. Rules: [case-study/README.md](../../../case-study/README.md). Reference values on the public leaderboard (2024): always 3926, accuracy 0.041; simple features + logistic regression, accuracy about 0.08; a TF-IDF text model (Session 13), accuracy above 0.8.

**What to expect.** The simple features describe the *form* of a description (length, numbers, lines, capital letters), not its content, and the language and country say where it was written. They cannot tell a toy from a lamp. This round is the baseline that shows why the text itself is needed; a low score is the expected result, not a failure.

Tasks for you are marked **Task**. Fitting takes about a minute per model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
while not (ROOT / "case-study").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "case-study" / "data"

# the 50,000-decision sample: logistic regression with 1,000 classes on all 309,529 decisions
# needs several GB of memory; the sample is enough for this baseline
train = pd.read_parquet(DATA / "train_sample.parquet").sort_values("start_date", ignore_index=True)
test = pd.read_parquet(DATA / "test.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
print(train.shape, test.shape)
print("headings in the sample:", train["heading"].nunique())
print("test columns:", list(test.columns))

## 1 · The simple features

The same definitions as `text_features.py` in the Session 2 workspace (`n_chars`, `n_words`, `n_lines`, `has_code`, `n_digits`, `upper_share`), plus the language and the issuing country. Each is a fixed rule applied to one decision at a time: no fitting, so no leakage, and the same function serves training and test data. Only columns that the test set also has are used: `keywords`, `classification_justification` and `cn_code` are written by customs and would leak the answer.

In [ ]:
def n_lines(text: str) -> int:
    return sum(1 for line in text.splitlines() if line.strip())

def upper_share(text: str) -> float:
    letters = [c for c in text if c.isalpha()]
    return sum(c.isupper() for c in letters) / len(letters) if letters else 0.0

def simple_features(d: pd.DataFrame) -> pd.DataFrame:
    t = d["description"]
    return pd.DataFrame({
        "log_chars": np.log(t.str.len()),                       # n_chars, on the log scale
        "log_words": np.log1p(t.str.split().str.len()),         # n_words
        "n_lines": t.map(n_lines),
        "has_code": t.str.contains("<CODE>", regex=False).astype(int),
        "n_digits": t.str.count(r"[0-9]"),
        "upper_share": t.map(upper_share),
        "language": d["language"],
        "country": d["issuing_country"],
    }, index=d.index)

X_tr, y_tr = simple_features(train), train["heading"]
X_te = simple_features(test)
numeric = ["log_chars", "log_words", "n_lines", "has_code", "n_digits", "upper_share"]
X_tr.groupby(y_tr.isin(["3926", "9503", "2106"]).map({True: "3926/9503/2106", False: "other"}))[numeric].mean().round(2)

## 2 · Validation that imitates the leaderboard

The leaderboard trains on the past and tests on the future, so we validate the same way: train on 2017–2021, validate on 2022–2023. `TimeSeriesSplit` would do the same several times; with 1,000 classes one split is enough for a baseline.

The preprocessing is part of the pipeline: the scaler and the one-hot encoder learn only from the training rows. Rare languages and countries (fewer than 20 decisions) are grouped by `min_frequency`.

In [ ]:
past = (train["start_date"].dt.year <= 2021).to_numpy()
pre = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20), ["language", "country"]),
])
candidates = {
    "always 3926": DummyClassifier(strategy="most_frequent"),
    "logistic regression": make_pipeline(pre, LogisticRegression(max_iter=300)),
}
predictions = {}
for name, model in candidates.items():
    model.fit(X_tr[past], y_tr[past])
    predictions[name] = pred = model.predict(X_tr[~past])
    print(f"{name:20s} accuracy {accuracy_score(y_tr[~past], pred):.3f}  "
          f"macro-F1 {f1_score(y_tr[~past], pred, average='macro'):.4f}  "
          f"distinct headings predicted {len(np.unique(pred))}")

In [ ]:
pred = pd.Series(predictions["logistic regression"], name="predicted")
truth = y_tr[~past].reset_index(drop=True)
summary = pd.DataFrame({
    "share of predictions": pred.value_counts(normalize=True),
    "share of true headings": truth.value_counts(normalize=True),
    "recall": truth.groupby(truth).apply(lambda s: (pred[s.index] == s).mean()),
}).sort_values("share of predictions", ascending=False).head(8)
summary["heading_description"] = nomenclature["heading_description"].reindex(summary.index).str[:50]
summary.round(3)

**Task 1.** Read the table. The model predicts only a few dozen of the most frequent headings. Why? Which features could separate 3926 (plastic articles) from 9503 (toys)? Why can no combination of length, digits, lines, language and country do so?

**Task 2.** Look at the probabilities: `candidates["logistic regression"].predict_proba(X_tr[~past]).max(axis=1)`. How confident is the model? Session 8, theory page 03, uses such confidences to route uncertain decisions to a human.

## 3 · Fit on all training data and write the submission

In [ ]:
l1 = make_pipeline(pre, LogisticRegression(max_iter=300)).fit(X_tr, y_tr)
submission = pd.DataFrame({"id": test["id"], "heading": l1.predict(X_te)})

# checks before uploading
assert list(submission.columns) == ["id", "heading"]
assert len(submission) == len(test) and submission["id"].is_unique
assert submission["heading"].str.fullmatch(r"\d{4}").all()      # four-digit strings, leading zeros kept

out = ROOT / "submission.csv"
submission.to_csv(out, index=False)
print("wrote", out, len(submission), "rows")
print(submission["heading"].value_counts(normalize=True).head(5).round(3).to_dict())

Upload `submission.csv` (in the repository root) to the course leaderboard. Only the lecturer has the hidden labels (`case-study/data/instructor/solution.csv`) and can score it from the repository root:

```bash
uv run python case-study/score.py submission.csv
```

The next cell does the same if the solution file is present (lecturer only).

In [ ]:
import importlib.util

solution = DATA / "instructor" / "solution.csv"
if solution.exists():
    spec = importlib.util.spec_from_file_location("score", ROOT / "case-study" / "score.py")
    score = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(score)
    result = score.score(pd.read_csv(out, dtype=str), pd.read_csv(solution, dtype=str))
    for k, v in result.items():
        print(f"{k:20s} {v:.4f}")
else:
    print("No solution file: submit to the leaderboard to get your score.")

**Task 3.** Record in your team log: features, model, validation accuracy and macro-F1 (2022–2023) and, once available, the public leaderboard score. Are they consistent? Compare with "always 3926".

**Task 4 (optional).** Add one simple feature of your own, computed with a fixed rule from the description (for example the share of digits, or whether the text contains a unit such as "cm" or "kg"). Re-run the validation and keep it only if the score improves by more than you would expect from chance (Session 7: bootstrap interval).

> **Do not** use `keywords`, `classification_justification`, `cn_code`, `chapter` or the end date as features: the test set does not have them, and they are written by customs during or after the classification (target leakage; see Session 7, theory page 03).

**Delete `submission.csv`** from the repository root after uploading it; it is a generated file and must not be committed.